# Week 11 · Notebook 3: Package B, rPPG + skin colour (Week 9 fusion + appearance)

**What this builds:** a deployable sex-prediction package from face video.
**Package B: rPPG + skin colour.** Package A's three parts plus gradient boosting on skin-colour features from the same video (per-region colour balance, brightness and brightness variability). Run Notebook 2 first: the shared parts are reused from `sex_deploy/cv_cache` and `final_components`, so only the skin-colour part is trained here.

**Two stages**
1. **Deployment tests (cross-validation).** The same folds as Weeks 9–10, grouped by person. Every model is trained on full 3-minute recordings of the training people. The test people are then scored as a deployed system would see them:
   - both sessions (before + after exercise), or **one session only**;
   - the **full recording, or only the first 120, 60 or 30 s**;
   - **with or without the quality gate** (Week 7: rejects sessions whose rPPG heart rate is probably wrong; needs no sensor at use time).
2. **The package.** Final models are trained on all 598 people and saved to `MCD_rPPG/sex_deploy/packages/rppg_plus_appearance/` with the shared code, settings, calibration and a model card. A last check loads the package from disk and predicts straight from traces and from raw videos.

**Calibration and decision.** The four (or three) model scores are averaged on the log-odds scale. A logistic calibration, fitted on cross-validation scores, turns that into P(male) for this cohort (one-third men). Two thresholds are stored: **balanced** (default; equal weight to men and women) and **accuracy** (most correct overall; favours "female").

**Needs:** Notebook 1 (`Week11_Extraction`) run first. GPU runtime.

**Run time on a Colab GPU:** about 15-30 min if Notebook 2 has run with the same settings (only the skin-colour part, the tests and the checks run); otherwise the same as Notebook 2.

In [ ]:
# ---- CONFIG: edit this cell only ----
import os
ROOT = os.environ.get('MCD_ROOT', '/content/drive/MyDrive/MCD_rPPG')
DEPLOY = os.environ.get('SEX_DEPLOY', f'{ROOT}/sex_deploy')
PACKAGE = 'rppg_plus_appearance'
COMPONENTS = ['cnn', 'rc', 'tab', 'app']
LENGTHS = ['full', '120s', '60s', '30s']   # must have been built by Notebook 1
CV_REPEATS = 2         # repeated 5-fold CV for the deployment tests
CNN_SEEDS = 2          # CNN copies per fold in CV
FINAL_CNN_SEEDS = 5    # CNN copies in the package
GATE_THRESHOLD = 0.20  # Week 7: reject a session if P(rPPG HR off by > 5 BPM) >= this
DEFAULT_THRESHOLD = 'balanced'   # 'balanced' or 'accuracy'
N_KERNELS = 500
EPOCHS, PATIENCE = 40, 8
N_BOOT = 300
N_VIDEO_CHECK = 3      # people predicted from raw videos in the final check (0 to skip)
QUICK = False          # tiny settings to check the notebook runs; results are not meaningful
if QUICK: LENGTHS, CV_REPEATS, CNN_SEEDS, FINAL_CNN_SEEDS, N_KERNELS, EPOCHS, PATIENCE, N_BOOT, N_VIDEO_CHECK = ['full', '30s'], 1, 1, 1, 100, 3, 2, 30, 1
if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# ---- shared code (sexpred_lib.py): written to Drive so every notebook and the deployed package use the same functions ----
LIB_SRC = r'''"""sexpred_lib: MCD-rPPG sex prediction, from face video to a calibrated probability.

Sources of each part, kept identical so deployed predictions match the training data:
- Video -> ROI colour traces: Week3_Stress_rPPG_600_FrontCam (3).ipynb (the notebook that wrote roi_traces on 2026-10-11).
- Session rPPG features (HRV, quality, skin colour): the same Week 3 notebook.
- 12-channel signals (POS, CHROM, green x 4 regions), appearance, pulse shape, random convolutions, CNN: Week 9.
- Quality gate: Week 7.
"""
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, resample_poly, welch
from scipy.stats import kurtosis, skew

LIB_VERSION = '1.0'
FS = 30.0
HR_BAND = (0.7, 3.5)
ROI_NAMES = ['face', 'forehead', 'left_cheek', 'right_cheek']
ROI_BOXES = {'face': (0.15, 0.85, 0.10, 0.90), 'forehead': (0.25, 0.75, 0.05, 0.25),
             'left_cheek': (0.12, 0.38, 0.50, 0.75), 'right_cheek': (0.62, 0.88, 0.50, 0.75)}
ALGOS = ['pos', 'chrom', 'green']
CHANNELS = [f'{r}_{a}' for r in ROI_NAMES for a in ALGOS]
POS_IDX = [CHANNELS.index(f'{r}_pos') for r in ROI_NAMES]
PHYS = ['rppg_hr', 'rppg_hr_beats', 'rppg_sdnn', 'rppg_rmssd', 'rppg_pnn50', 'rppg_ln_sdnn', 'rppg_ln_rmssd', 'rppg_ln_lf', 'rppg_ln_hf',
        'rppg_lf_hf', 'rppg_rise_frac', 'rppg_half_width']
QUAL = ['rppg_snr', 'rppg_snr_face', 'rppg_snr_forehead', 'rppg_snr_left_cheek', 'rppg_snr_right_cheek', 'rppg_hr_win_sd',
        'rppg_hr_consistency', 'rppg_n_beats', 'rppg_ibi_ok_frac', 'rppg_beat_consistency']
SKIN = ['rppg_skin_r', 'rppg_skin_g', 'rppg_skin_lum']
MORPH_COLS = ['rise_frac', 'w25', 'w50', 'w75', 'max_slope_pos', 'slope_ratio', 'area_ratio', 'skew', 'kurt', 'apg_b_a', 'apg_late_a',
              'h2_h1', 'h3_h1', 'h4_h1', 'consistency', 'kept_frac', 'beats_per_min']
SHAPE_COLS = [f'shape_{c}' for c in MORPH_COLS] + [f'snr8_{r}' for r in ROI_NAMES]
TAB_COLS = PHYS + QUAL + SHAPE_COLS                       # rPPG tabular model
APP_COLS = [f'{r}_{s}' for r in ROI_NAMES for s in ('chroma_r', 'chroma_g', 'lum', 'lum_var')] + SKIN   # appearance model
GATE_COLS = ['rppg_face_missing_frac', 'rppg_fps_measured', 'rppg_snr_face', 'rppg_snr_forehead', 'rppg_snr_left_cheek',
             'rppg_snr_right_cheek', 'rppg_snr', 'rppg_hr_win_sd', 'rppg_hr_consistency', 'rppg_n_beats', 'rppg_ibi_ok_frac',
             'rppg_beat_consistency', 'rppg_skin_lum', 'hr_beat_gap']
_trapz = getattr(np, 'trapezoid', None) or np.trapz
WIN = int(20 * FS)   # CNN and random-conv window: 20 s

# =====================================================================================
# 1. Video -> ROI colour traces (Week 3 extractor, unchanged)
# =====================================================================================
def cascade_path(tmp_dir='/tmp'):
    import cv2
    name = 'haarcascade_frontalface_default.xml'
    root = getattr(getattr(cv2, 'data', None), 'haarcascades', None)
    if root and (Path(root) / name).exists():
        return str(Path(root) / name)
    dst = Path(tmp_dir) / name
    if not dst.exists():
        import urllib.request
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/{name}', dst)
    return str(dst)

def extract_roi_traces(video_path, out_path=None, detect_every=10, ema=0.3, scale=0.5, cascade=None):
    """Per-frame skin-masked mean RGB for face, forehead, left cheek and right cheek. Returns the same keys as roi_traces/*.npz."""
    import cv2
    cv2.setNumThreads(1)
    det = cv2.CascadeClassifier(cascade or cascade_path())
    if det.empty():
        raise RuntimeError('Haar face detector failed to load (OpenCV build without it?)')
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'cannot open video {video_path}')
    fps_c = cap.get(cv2.CAP_PROP_FPS)
    traces, dup, found, box, prev_small = [], [], [], None, None
    i = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        small = cv2.resize(bgr, (64, 48), interpolation=cv2.INTER_AREA)
        dup.append(prev_small is not None and np.array_equal(small, prev_small)); prev_small = small
        if i % detect_every == 0:
            g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), None, fx=scale, fy=scale)
            faces = det.detectMultiScale(g, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
            d = None
            if len(faces):
                x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
                d = np.array([x, y, w, h], float) / scale
                box = d if box is None else (1 - ema) * box + ema * d
            found.append(d is not None)
        else:
            found.append(found[-1] if found else False)
        row = np.full((len(ROI_NAMES), 3), np.nan, np.float32)
        if box is not None:
            x, y, w, h = box
            rgb = bgr[..., ::-1]
            for k, n in enumerate(ROI_NAMES):
                fx0, fx1, fy0, fy1 = ROI_BOXES[n]
                xa, xb = int(max(0, x + fx0 * w)), int(min(bgr.shape[1], x + fx1 * w))
                ya, yb = int(max(0, y + fy0 * h)), int(min(bgr.shape[0], y + fy1 * h))
                if xb - xa < 4 or yb - ya < 4:
                    continue
                patch = rgb[ya:yb, xa:xb]
                ycc = cv2.cvtColor(bgr[ya:yb, xa:xb], cv2.COLOR_BGR2YCrCb)
                skin = (ycc[..., 1] >= 133) & (ycc[..., 1] <= 173) & (ycc[..., 2] >= 77) & (ycc[..., 2] <= 127)
                px = patch[skin] if skin.sum() > 0.2 * skin.size else patch.reshape(-1, 3)
                row[k] = px.mean(0)
        traces.append(row); i += 1
    cap.release()
    tr = dict(traces=np.array(traces, np.float32), roi_names=np.array(ROI_NAMES), dup=np.array(dup), face_found=np.array(found),
              fps_container=fps_c)
    if out_path is not None:
        np.savez_compressed(out_path, **tr)
    return tr

def load_traces(path):
    with np.load(path) as z:
        return {k: z[k] for k in z.files}

TS_RE = re.compile(r'(\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:[.,]\d+)?)')
def parse_meta_timestamps(path):
    """Per-frame timestamps (seconds from start) from an MCD meta file, or None."""
    if path is None or not Path(path).exists():
        return None
    lines = Path(path).read_text(errors='ignore').splitlines()
    ts = [pd.Timestamp(m.group(1).replace(',', '.')).value / 1e9 for m in map(TS_RE.search, lines) if m]
    if len(ts) < 100:
        ts = []
        for ln in lines:
            nums = re.findall(r'[-+]?\d+(?:\.\d+)?', ln)
            if len(nums) == 1:
                ts.append(float(nums[0]))
    if len(ts) < 100:
        return None
    t = np.asarray(ts, float); t = t - t[0]; d = np.diff(t)
    if np.mean(d < 0) > 0.01:
        return None
    if np.median(d) > 1:
        t = t / 1000.0
    return t

def time_base(N, ts, fps_hdr):
    """Week 3 rule: meta timestamps if they match the frame count, else the meta fps, else the container fps."""
    ts = np.array([]) if ts is None else np.asarray(ts, float)
    if len(ts) >= 100 and abs(len(ts) - N) <= 5:
        t = ts[:N]
        if len(t) < N:
            t = np.concatenate([t, t[-1] + np.arange(1, N - len(t) + 1) * np.median(np.diff(t))])
        return t, 1
    if len(ts) >= 100:
        return np.arange(N) / ((len(ts) - 1) / (ts[-1] - ts[0])), 2
    fps = fps_hdr if 5 < fps_hdr < 121 else 30.0
    return np.arange(N) / fps, 0

def _prepared(tr, ts, max_seconds=None):
    rgb = tr['traces'].astype(float); dup = tr['dup'].astype(bool); N = len(rgb)
    t, used = time_base(N, ts, float(tr['fps_container']))
    if max_seconds is not None:
        n = int(np.searchsorted(t, t[0] + max_seconds, side='right'))
        rgb, dup, t = rgb[:n], dup[:n], t[:n]
    return rgb, dup, t, used

# =====================================================================================
# 2. Week 3 session features (rPPG part)
# =====================================================================================
def bandpass(x, fs, lo, hi, order=3):
    b, a = butter(order, [lo / (fs / 2), hi / (fs / 2)], btype='band'); return filtfilt(b, a, x)

def bandpass_nd(x, fs, lo, hi, order=3):
    b, a = butter(order, [lo / (fs / 2), min(hi / (fs / 2), .95)], btype='band'); return filtfilt(b, a, x, axis=-1)

def interp_nans(a):
    a = np.array(a, float, copy=True); flat = a.reshape(len(a), -1); idx = np.arange(len(a))
    for j in range(flat.shape[1]):
        m = np.isfinite(flat[:, j])
        if m.sum() < 0.5 * len(m):
            return None
        flat[~m, j] = np.interp(idx[~m], idx[m], flat[m, j])
    return flat.reshape(a.shape)

def _windows(rgb, fs, win):
    from numpy.lib.stride_tricks import sliding_window_view
    l = int(round(win * fs)); W = sliding_window_view(rgb, l, axis=0); mu = W.mean(axis=2, keepdims=True); mu[mu == 0] = 1e-9
    return W / mu, l

def _overlap_add(h, N, l):
    h = h - h.mean(axis=1, keepdims=True); H = np.zeros(N)
    for j in range(l):
        H[j:j + len(h)] += h[:, j]
    return H

def pos(rgb, fs, win=1.6):
    """POS (Wang et al. 2017). Vectorised; identical output to the Week 3 loop version."""
    C, l = _windows(rgb, fs, win); S0 = C[:, 1] - C[:, 2]; S1 = -2 * C[:, 0] + C[:, 1] + C[:, 2]; sd1 = S1.std(axis=1, keepdims=True)
    return _overlap_add(S0 + (S0.std(axis=1, keepdims=True) / np.where(sd1 > 0, sd1, np.inf)) * S1, len(rgb), l)

def chrom(rgb, fs, win=1.6):
    C, l = _windows(rgb, fs, win); X = 3 * C[:, 0] - 2 * C[:, 1]; Y = 1.5 * C[:, 0] + C[:, 1] - 1.5 * C[:, 2]; sy = Y.std(axis=1, keepdims=True)
    return _overlap_add(X - (X.std(axis=1, keepdims=True) / np.where(sy > 0, sy, np.inf)) * Y, len(rgb), l)

def green_norm(rgb, fs, win=1.6):
    l = int(round(win * fs)); g = rgb[:, 1]; m = np.convolve(g, np.ones(l) / l, mode='same'); return g / np.where(m == 0, 1e-9, m) - 1

def spectrum(x, fs, seg_s=12):
    nper = int(min(len(x), seg_s * fs)); return welch(x, fs, nperseg=nper, noverlap=nper // 2, nfft=max(8192, nper))

def hr_harmonic(x, fs, band=HR_BAND):
    f, P = spectrum(x, fs); m = (f >= band[0]) & (f <= band[1])
    score = P[m] + 0.5 * np.interp(2 * f[m], f, P, right=0); return 60 * f[m][np.argmax(score)], f, P

def snr_db(f, P, f0, half=0.1):
    sig = (np.abs(f - f0) <= half) | (np.abs(f - 2 * f0) <= half); rng = (f >= 0.6) & (f <= 4.0)
    num, den = P[sig & rng].sum(), P[~sig & rng].sum()
    return 10 * np.log10(num / den) if num > 0 and den > 0 else np.nan

def detect_beats(x, fs, hr_bpm):
    period = fs * 60.0 / hr_bpm; pk, _ = find_peaks(x, distance=max(1, int(0.6 * period)), prominence=0.1 * np.std(x)); return pk

def hrv_from_peaks(pk, fs, min_dur_freq=100):
    t = pk / fs; ibi = np.diff(t)
    if len(ibi) < 10:
        return {}
    tb = t[1:]; med = pd.Series(ibi).rolling(7, center=True, min_periods=1).median().values
    ok = (ibi > 0.33) & (ibi < 1.5) & (np.abs(ibi - med) < 0.2 * med)
    if ok.sum() < 10:
        return dict(ibi_ok_frac=ok.mean())
    ic, tc = ibi[ok], tb[ok]; d = np.diff(ibi)[ok[1:] & ok[:-1]]
    o = dict(n_beats=len(pk), ibi_ok_frac=ok.mean(), hr_beats=60 / ic.mean(), sdnn=1000 * ic.std(ddof=1),
             rmssd=1000 * np.sqrt(np.mean(d ** 2)) if len(d) else np.nan, pnn50=np.mean(np.abs(d) > 0.05) if len(d) else np.nan)
    o['ln_sdnn'] = np.log(max(o['sdnn'], 1e-3)); o['ln_rmssd'] = np.log(max(o['rmssd'], 1e-3)) if np.isfinite(o['rmssd']) else np.nan
    if tc[-1] - tc[0] >= min_dur_freq and len(ic) >= 60:
        g = np.arange(tc[0], tc[-1], 0.25); y = np.interp(g, tc, ic); y = y - y.mean(); f, P = welch(y, 4.0, nperseg=min(256, len(y)))
        lf = _trapz(P[(f >= 0.04) & (f < 0.15)], f[(f >= 0.04) & (f < 0.15)]) * 1e6; hf = _trapz(P[(f >= 0.15) & (f < 0.40)], f[(f >= 0.15) & (f < 0.40)]) * 1e6
        o.update(lf=lf, hf=hf, ln_lf=np.log(max(lf, 1e-6)), ln_hf=np.log(max(hf, 1e-6)), lf_hf=lf / hf if hf > 0 else np.nan)
    return o

def morphology(x, pk, fs):
    if len(pk) < 10:
        return {}
    T = int(np.median(np.diff(pk))); a, b = int(0.6 * T), int(0.4 * T)
    segs = [x[p - a:p + b] for p in pk if p - a >= 0 and p + b <= len(x)]
    if len(segs) < 10:
        return {}
    S = np.array(segs); S = (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-12); tpl = S.mean(0)
    cons = np.mean([np.corrcoef(s, tpl)[0, 1] for s in S]); i_min = int(np.argmin(tpl[:a + 1])); half = (tpl.max() + tpl.min()) / 2
    return dict(beat_consistency=cons, rise_frac=(a - i_min) / T, half_width=np.mean(tpl > half))

def windowed_hr(x, fs, win=10, step=2):
    n, s = int(win * fs), int(step * fs); return np.array([hr_harmonic(x[i:i + n], fs)[0] for i in range(0, len(x) - n + 1, s)])

def pulse_features(x, fs):
    hr, f, P = hr_harmonic(x, fs); o = dict(hr=hr, snr=snr_db(f, P, hr / 60)); w = windowed_hr(x, fs)
    if len(w):
        o['hr_win_sd'] = np.std(w); o['hr_consistency'] = np.mean(np.abs(w - hr) < 5)
    pk = detect_beats(x, fs, hr); o.update(hrv_from_peaks(pk, fs)); o.update(morphology(x, pk, fs)); return o

def rppg_features(tr, ts=None, max_seconds=None):
    """Week 3 rppg_features (POS per ROI, 0.7-3.5 Hz, SNR-weighted combination). Keys carry the rppg_ prefix."""
    rgb, dup, t, used = _prepared(tr, ts, max_seconds); N = len(rgb)
    names = [str(n) for n in tr['roi_names']] if 'roi_names' in tr else ROI_NAMES
    o = dict(face_missing_frac=float(np.mean(~np.isfinite(rgb[:, 0, 0]))), dup_frac=float(dup.mean()))
    rgb = rgb.copy(); rgb[dup] = np.nan
    o['used_ts'] = used; o['fps_measured'] = (N - 1) / (t[-1] - t[0]); o['duration_s'] = t[-1] - t[0]
    tu = np.arange(t[0], t[-1], 1 / FS); sigs, snrs = [], []
    for k, name in enumerate(names):
        c = interp_nans(rgb[:, k, :])
        if c is None:
            continue
        cu = np.column_stack([np.interp(tu, t, c[:, j]) for j in range(3)])
        s = bandpass(pos(cu, FS), FS, *HR_BAND); hr_k, f, P = hr_harmonic(s, FS); snr_k = snr_db(f, P, hr_k / 60)
        o[f'snr_{name}'] = snr_k; sigs.append(s / (np.std(s) + 1e-12)); snrs.append(snr_k if np.isfinite(snr_k) else -20)
        if name == 'face':
            m = c.mean(0); o.update(skin_r=m[0] / m.sum(), skin_g=m[1] / m.sum(), skin_lum=m.mean())
    if sigs:
        w = 10 ** (np.array(snrs) / 10); comb = np.sum([wi * si for wi, si in zip(w, sigs)], axis=0) / w.sum()
        o.update(pulse_features(comb, FS))
    o = {f'rppg_{k}': v for k, v in o.items()}
    if 'rppg_hr' in o and 'rppg_hr_beats' in o:
        o['hr_beat_gap'] = abs(o['rppg_hr'] - o['rppg_hr_beats'])
    return o

def load_pw(path):
    vals = []
    with open(path, errors='ignore') as fh:
        for ln in fh:
            tok = ln.split()
            if tok:
                try: vals.append(float(tok[0]))
                except ValueError: pass
    return np.asarray(vals)

def contact_hr(pw_path, fs=100.0, max_seconds=None):
    """Contact-PPG heart rate (Week 3), optionally over the first max_seconds. Used only as the label for training the quality gate."""
    from scipy.signal import detrend
    x = load_pw(pw_path)
    if max_seconds is not None:
        x = x[:int(max_seconds * fs)]
    if len(x) < min(30, max_seconds or 30) * fs - 1:
        return np.nan
    return hr_harmonic(bandpass(detrend(x), fs, *HR_BAND), fs)[0]

# =====================================================================================
# 3. Week 9 signals: 4 regions x (POS, CHROM, green), appearance, pulse shape
# =====================================================================================
def multi_signals(tr, ts=None, max_seconds=None):
    """12 rPPG channels at 30 Hz, 0.3-12 Hz, z-scored (float16), and 16 appearance values. None if a region is mostly missing."""
    rgb, dup, t, _ = _prepared(tr, ts, max_seconds); rgb = rgb.copy(); rgb[dup] = np.nan
    tu = np.arange(t[0], t[-1], 1 / FS); sig, app = [], []
    for k in range(rgb.shape[1]):
        c = interp_nans(rgb[:, k, :])
        if c is None:
            return None, None
        cu = np.column_stack([np.interp(tu, t, c[:, j]) for j in range(3)])
        for fn in (pos, chrom, green_norm):
            s = bandpass_nd(fn(cu, FS), FS, .3, 12.); sig.append(s / (s.std() + 1e-12))
        tot = cu.sum(1) + 1e-9; lum = cu.mean(1); sec = lum[:len(lum) // 30 * 30].reshape(-1, 30).mean(1)
        app += [(cu[:, 0] / tot).mean(), (cu[:, 1] / tot).mean(), lum.mean(), sec.std() / (sec.mean() + 1e-9)]
    return np.stack(sig).astype(np.float16), np.array(app, np.float32)

def views(multi_raw):
    """The two model inputs from the stored 12 channels: all 12 at 0.5-8 Hz (CNN) and the 4 POS channels at 0.5-8 Hz (random conv, pulse shape)."""
    a = multi_raw.astype(np.float32)
    return bandpass_nd(a, FS, .5, 8.).astype(np.float32), bandpass_nd(a[POS_IDX], FS, .5, 8.).astype(np.float32)

def hr_est(x, fs=FS):
    y = bandpass_nd(x, fs, .7, 3.5); f, Pw = welch(y, fs, nperseg=min(len(y), int(12 * fs)), nfft=8192)
    m = (f >= .7) & (f <= 3.5); sc = Pw[m] + .5 * np.interp(2 * f[m], f, Pw, right=0); return f[m][np.argmax(sc)]

def snr_db8(x, f0, fs=FS):
    f, Pw = welch(bandpass_nd(x, fs, .5, 4.5), fs, nperseg=min(len(x), int(12 * fs)), nfft=8192)
    sig = ((np.abs(f - f0) <= .1) | (np.abs(f - 2 * f0) <= .1)) & (f >= .6) & (f <= 4.); rng = (f >= .6) & (f <= 4.)
    return 10 * np.log10(Pw[sig].sum() / max(Pw[rng & ~sig].sum(), 1e-12))

def combine(a):
    snr = np.array([snr_db8(ch, hr_est(ch)) for ch in a]); w = 10 ** (np.clip(np.nan_to_num(snr, nan=-20), -20, 20) / 10)
    x = (w[:, None] * a).sum(0) / w.sum(); return x / (x.std() + 1e-12), snr

def pulse_template(x, fs=FS, up=4, n=64):
    f0 = hr_est(x, fs); xu = resample_poly(x, up, 1); xn = resample_poly(bandpass_nd(x, fs, .7, 3.5), up, 1); T = fs * up / f0
    pk, _ = find_peaks(xn, distance=max(1, int(.6 * T))); a, b = int(.45 * T), int(.55 * T)
    segs = np.array([xu[p - a:p + b] for p in pk if p - a >= 0 and p + b <= len(xu)])
    if len(segs) < 10:
        return None
    segs = (segs - segs.mean(1, keepdims=True)) / (segs.std(1, keepdims=True) + 1e-9); med = np.median(segs, 0)
    r = np.array([np.corrcoef(s, med)[0, 1] for s in segs]); keep = r > .5
    if keep.sum() < 10:
        keep = np.ones(len(segs), bool)
    tpl = segs[keep].mean(0); tpl = np.roll(tpl, -int(np.argmin(tpl[:int(.7 * len(tpl))])))
    t = np.interp(np.linspace(0, len(tpl) - 1, n), np.arange(len(tpl)), tpl); t = (t - t.mean()) / (t.std() + 1e-9)
    d1 = np.gradient(t); d2 = np.gradient(d1); pkp = int(np.argmax(t)); rng_ = t.max() - t.min(); base = t - t.min()
    ia = int(np.argmax(d2[:max(pkp, 2)])); ib = ia + int(np.argmin(d2[ia:max(pkp + 4, ia + 2)])); late = d2[pkp:int(.75 * n)]
    f, Pw = welch(x, fs, nperseg=min(len(x), int(12 * fs)), nfft=8192); h = [Pw[np.abs(f - k * f0) <= .08].sum() + 1e-12 for k in (1, 2, 3, 4)]
    return dict(rise_frac=pkp / n, w25=np.mean(t > t.min() + .75 * rng_), w50=np.mean(t > t.min() + .5 * rng_), w75=np.mean(t > t.min() + .25 * rng_),
                max_slope_pos=np.argmax(d1) / n, slope_ratio=d1.max() / max(-d1.min(), 1e-9), area_ratio=base[:pkp + 1].sum() / max(base[pkp:].sum(), 1e-9),
                skew=skew(t), kurt=kurtosis(t), apg_b_a=d2[ib] / max(abs(d2[ia]), 1e-9), apg_late_a=(late.max() if len(late) else np.nan) / max(abs(d2[ia]), 1e-9),
                h2_h1=np.log10(h[1] / h[0]), h3_h1=np.log10(h[2] / h[0]), h4_h1=np.log10(h[3] / h[0]), consistency=r[keep].mean(), kept_frac=keep.mean(),
                beats_per_min=60 * f0)

def shape_features(pos4):
    """Pulse-shape features of the SNR-weighted average of the 4 POS channels, plus each region's SNR."""
    out = {c: np.nan for c in SHAPE_COLS}
    try:
        x, snr = combine(pos4)
        for r, s in zip(ROI_NAMES, snr): out[f'snr8_{r}'] = s
        f = pulse_template(x)
        if f is not None:
            out.update({f'shape_{k}': v for k, v in f.items()})
    except Exception:
        pass
    return out

# =====================================================================================
# 4. Everything one session needs, from traces
# =====================================================================================
def session_record(tr, ts=None, max_seconds=None):
    """All features of one session (rPPG, pulse shape, appearance) and the stored 12-channel signal (float16), or None if unusable."""
    feats = rppg_features(tr, ts, max_seconds)
    raw, app = multi_signals(tr, ts, max_seconds)
    if raw is None:
        return feats, None
    feats.update(shape_features(views(raw)[1])); feats.update(dict(zip(APP_COLS[:16], app)))
    return feats, raw

def session_inputs(tr, ts=None, max_seconds=None):
    """Returns (features dict, 12-channel CNN input, 4 POS channels for random conv). The arrays are None if the session is unusable."""
    feats, raw = session_record(tr, ts, max_seconds)
    if raw is None:
        return feats, None, None
    m12, p4 = views(raw)
    return feats, m12, p4

def usable(feats, raw):
    return raw is not None and np.isfinite(feats.get('rppg_hr', np.nan)) and raw.shape[-1] >= WIN

# =====================================================================================
# 5. Random convolutions (Week 9)
# =====================================================================================
_KERN = {}
def kernels(L, n_kernels, seed=0):
    key = (L, n_kernels, seed)
    if key not in _KERN:
        rg = np.random.default_rng(seed); nfft = 2 * L; ks, dil = [], []
        for _ in range(n_kernels):
            k = rg.normal(size=rg.choice([7, 9, 11])); k -= k.mean(); d = int(2 ** rg.uniform(0, np.log2((L - 1) / (len(k) - 1)) * .55))
            kk = np.zeros((len(k) - 1) * d + 1); kk[::d] = k; ks.append(np.fft.rfft(kk, nfft)); dil.append(len(kk))
        _KERN[key] = (ks, dil)
    return _KERN[key]

def rand_features(sigs, n_kernels=500, win_s=20, n_win=6):
    """sigs: list of 1-D signals (or None). Returns (len(sigs), 2*n_kernels) array, NaN rows for unusable signals."""
    L = int(FS * win_s); nfft = 2 * L; ks, dil = kernels(L, n_kernels); windows, owner = [], []
    for i, x in enumerate(sigs):
        if x is None or len(x) < L:
            continue
        x = np.nan_to_num(np.asarray(x, np.float32))
        for s in np.linspace(0, len(x) - L, n_win).astype(int):
            w = x[s:s + L]; windows.append((w - w.mean()) / (w.std() + 1e-6)); owner.append(i)
    out = np.full((len(sigs), 2 * n_kernels), np.nan, np.float32)
    if not windows:
        return out
    F = np.fft.rfft(np.stack(windows), nfft, axis=1); feat = np.zeros((len(windows), 2 * n_kernels), np.float32)
    for j, (K, l) in enumerate(zip(ks, dil)):
        c = np.fft.irfft(F * K, nfft, axis=1)[:, l - 1:L]; feat[:, 2 * j] = (c > 0).mean(1); feat[:, 2 * j + 1] = c.max(1)
    g = pd.DataFrame(feat).groupby(np.array(owner)).mean()
    out[g.index.values] = g.values
    return out

def rc_session_features(pos4_list, n_kernels=500):
    """Random-conv features for many sessions: each region's POS channel separately, concatenated (4 x 2 x n_kernels)."""
    return np.hstack([rand_features([None if a is None else a[j] for a in pos4_list], n_kernels) for j in range(len(POS_IDX))])

# =====================================================================================
# 6. CNN (Week 9 ResNet1D, improved training)
# =====================================================================================
def _torch():
    import torch, torch.nn as nn
    return torch, nn

_NET = None
def make_net(in_ch):
    global _NET
    torch, nn = _torch()
    if _NET is None:
        class ResBlock(nn.Module):
            def __init__(self, i, o, stride=2, k=15):
                super().__init__()
                self.net = nn.Sequential(nn.Conv1d(i, o, k, stride=stride, padding=k // 2), nn.BatchNorm1d(o), nn.ReLU(), nn.Dropout(.1),
                                         nn.Conv1d(o, o, k, padding=k // 2), nn.BatchNorm1d(o))
                self.skip = nn.Sequential(nn.Conv1d(i, o, 1, stride=stride), nn.BatchNorm1d(o)); self.act = nn.ReLU()
            def forward(self, x): return self.act(self.net(x) + self.skip(x))
        class Net(nn.Module):
            def __init__(self, in_ch):
                super().__init__()
                self.enc = nn.Sequential(nn.Conv1d(in_ch, 32, 15, stride=2, padding=7), nn.BatchNorm1d(32), nn.ReLU(), ResBlock(32, 32), ResBlock(32, 64), ResBlock(64, 64))
                self.head = nn.Sequential(nn.Dropout(.4), nn.Linear(64, 1))
            def forward(self, x): return self.head(self.enc(x).mean(-1)).squeeze(-1)
        _NET = Net
    return _NET(in_ch)

def zwin(x): return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-6)

def _sample(items, sig, flag, rng, ch_group=3):
    C = sig[items[0]].shape[0]; X = np.empty((len(items), C + 1, WIN), np.float32); n_groups = C // ch_group
    for n, i in enumerate(items):
        T = sig[i].shape[1]; L = min(T, int(round(WIN * rng.uniform(.92, 1.08)))); st = rng.integers(0, T - L + 1); seg = sig[i][:, st:st + L].astype(np.float32)
        if L != WIN:
            x = np.linspace(0, L - 1, WIN); i0 = np.floor(x).astype(int); w = (x - i0).astype(np.float32); i1 = np.minimum(i0 + 1, L - 1); seg = seg[:, i0] * (1 - w) + seg[:, i1] * w
        seg = zwin(seg); seg = seg + rng.normal(0, rng.uniform(0, .3), seg.shape).astype(np.float32)
        if n_groups > 1 and rng.random() < .3:
            g = rng.integers(n_groups); seg[g * ch_group:(g + 1) * ch_group] = 0
        X[n, :C] = seg; X[n, C] = flag[i]
    return X

def cnn_session_logit(model, x, flag, device, stride_s=5):
    """Mean window logit for one session (x: channels x samples). NaN if shorter than 20 s."""
    torch, _ = _torch()
    if x is None or x.shape[1] < WIN:
        return np.nan
    st = np.arange(0, x.shape[1] - WIN + 1, int(stride_s * FS)); C = x.shape[0]; X = np.empty((len(st), C + 1, WIN), np.float32)
    for m, s in enumerate(st): X[m, :C] = zwin(x[:, s:s + WIN]); X[m, C] = flag
    model.eval()
    with torch.no_grad():
        return float(model(torch.from_numpy(X).to(device)).mean().cpu())

def train_cnn(sig, flag, y, ids, train_idx, seed, device, epochs=40, patience=8, per_session=8, batch=128, ch_group=3):
    """Week 9 improved training. sig: list of (12, T) arrays or None; train_idx: session indices. Early stopping on 10% of training people."""
    torch, nn = _torch()
    rng = np.random.default_rng(seed); torch.manual_seed(seed); people = pd.Series(y[train_idx], index=ids[train_idx]).groupby(level=0).first()
    val_ids = set(np.concatenate([rng.choice(people.index[people == c], max(3, int(.1 * (people == c).sum())), replace=False) for c in (0, 1)]))
    usable = [i for i in train_idx if sig[i] is not None and sig[i].shape[1] >= WIN]
    fit = np.array([i for i in usable if ids[i] not in val_ids]); val = np.array([i for i in usable if ids[i] in val_ids])
    model = make_net(sig[fit[0]].shape[0] + 1).to(device); pos_ = float(y[fit].mean()); pw = torch.tensor((1 - pos_) / pos_, dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    steps = epochs * int(np.ceil(len(fit) * per_session / batch)); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(steps, 1))
    best, state, bad = -1., None, 0
    from sklearn.metrics import roc_auc_score
    for ep in range(epochs):
        model.train(); order = rng.permutation(np.repeat(fit, per_session))
        for b in range(0, len(order), batch):
            ch = order[b:b + batch]
            if len(ch) < 2: continue
            X = torch.from_numpy(_sample(ch, sig, flag, rng, ch_group)).to(device); t = torch.from_numpy((y[ch] * .9 + .05).astype(np.float32)).to(device)
            loss = nn.functional.binary_cross_entropy_with_logits(model(X), t, pos_weight=pw); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        pv = np.array([cnn_session_logit(model, sig[i], flag[i], device) for i in val])
        f = pd.DataFrame(dict(i=ids[val], y=y[val], p=pv)).dropna().groupby('i').mean(); auc = roc_auc_score(f.y.astype(int), f.p)
        if auc > best + 1e-4: best, bad, state = auc, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval(); return model

# =====================================================================================
# 7. Tabular models and the quality gate
# =====================================================================================
def make_hgb():
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    return make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_iter=200, max_depth=3, learning_rate=.04, min_samples_leaf=15,
                         l2_regularization=1., class_weight='balanced', random_state=0))

def make_rc_model():
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=.001, class_weight='balanced', max_iter=3000))

def make_gate():
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    return make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_depth=3, max_iter=150, learning_rate=.05, random_state=0))

def two_session_row(before, after, cols):
    """One person row: before, after and change (after minus before) for each column."""
    b = np.array([before.get(c, np.nan) if before else np.nan for c in cols], float); a = np.array([after.get(c, np.nan) if after else np.nan for c in cols], float)
    return np.concatenate([b, a, a - b])

def single_session_row(sess, is_after, cols):
    return np.concatenate([np.array([sess.get(c, np.nan) for c in cols], float), [float(is_after)]])

def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def sigmoid(z): return 1 / (1 + np.exp(-np.asarray(z, float)))

# =====================================================================================
# 8. Deployed predictor
# =====================================================================================
class SexPredictor:
    """Loads a package folder written by the package notebook and predicts from videos or ROI traces.

    predict(before_video=..., after_video=..., before_meta=..., after_meta=...)  # either session may be omitted
    Returns a dict with probability_male, prediction ('M', 'F' or None if no session passed the quality gate), mode, components and per-session quality.
    """
    def __init__(self, package_dir, device=None):
        import joblib
        self.dir = Path(package_dir); self.cfg = json.loads((self.dir / 'config.json').read_text())
        self.comp = self.cfg['components']; self.m = {}
        for name in ('rc', 'tab_two', 'tab_single', 'app_two', 'app_single', 'gate', 'calibrators'):
            p = self.dir / f'{name}.joblib'
            if p.exists(): self.m[name] = joblib.load(p)
        self.cnn = []
        if 'cnn' in self.comp:
            torch, _ = _torch(); self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
            for p in sorted(self.dir.glob('cnn_seed*.pt')):
                net = make_net(len(CHANNELS) + 1); net.load_state_dict(torch.load(p, map_location=self.device)); net.to(self.device).eval(); self.cnn.append(net)

    def session_from_video(self, video, meta=None, step='before', max_seconds=None):
        return self.session_from_traces(extract_roi_traces(video), parse_meta_timestamps(meta), step, max_seconds)

    def session_from_traces(self, tr, ts=None, step='before', max_seconds=None):
        f, m12, p4 = session_inputs(tr, ts, max_seconds)
        return dict(step=step, feats=f, m12=m12, p4=p4)

    def score_sessions(self, sessions, use_gate=True, threshold_kind=None):
        cfg = self.cfg; out = dict(sessions=[], components={}, package=cfg['package'], lib_version=LIB_VERSION)
        kept = []
        for s in sessions:
            f = s['feats']; q = dict(step=s['step'], usable=bool(usable(f, s['m12'])), duration_s=f.get('rppg_duration_s'))
            if q['usable'] and 'gate' in self.m:
                x = np.array([[f.get(c, np.nan) for c in GATE_COLS]], float); q['p_bad'] = float(self.m['gate'].predict_proba(x)[0, 1])
                q['passed_gate'] = q['p_bad'] < cfg['gate_threshold']
            else:
                q['passed_gate'] = q['usable']
            if q['usable'] and (q['passed_gate'] or not use_gate): kept.append(s)
            out['sessions'].append(q)
        if not kept:
            out.update(probability_male=None, prediction=None, mode=None, reason='no session passed the quality check'); return out
        steps = {s['step']: s for s in kept}; two = 'before' in steps and 'after' in steps; mode = 'two' if two else 'single'
        lg = []
        if 'cnn' in self.comp:
            ps = [sigmoid(np.nanmean([cnn_session_logit(net, s['m12'], float(s['step'] == 'after'), self.device) for net in self.cnn])) for s in kept]
            out['components']['cnn'] = float(np.nanmean(ps))
        if 'rc' in self.comp:
            X = rc_session_features([s['p4'] for s in kept], cfg['n_kernels']); out['components']['rc'] = float(np.mean(self.m['rc'].predict_proba(X)[:, 1]))
        for comp, cols in (('tab', TAB_COLS), ('app', APP_COLS)):
            if comp not in self.comp: continue
            if two:
                x = two_session_row(steps['before']['feats'], steps['after']['feats'], cols)[None]; p = self.m[f'{comp}_two'].predict_proba(x)[0, 1]
            else:
                p = float(np.mean([self.m[f'{comp}_single'].predict_proba(single_session_row(s['feats'], s['step'] == 'after', cols)[None])[0, 1] for s in kept]))
            out['components'][comp] = float(p)
        z = float(np.mean(logit(list(out['components'].values()))))
        cal = self.m['calibrators'][mode]; prob = float(cal.predict_proba(np.array([[z]]))[0, 1])
        kind = threshold_kind or cfg['default_threshold']; thr = cfg['thresholds'][mode][kind]
        out.update(fused_logit=z, probability_male=prob, threshold=thr, threshold_kind=kind, prediction='M' if prob >= thr else 'F', mode=mode,
                   sessions_used=[s['step'] for s in kept])
        return out

    def predict(self, before_video=None, after_video=None, before_meta=None, after_meta=None, use_gate=True, threshold_kind=None):
        ss = []
        if before_video is not None: ss.append(self.session_from_video(before_video, before_meta, 'before'))
        if after_video is not None: ss.append(self.session_from_video(after_video, after_meta, 'after'))
        if not ss: raise ValueError('give at least one video')
        return self.score_sessions(ss, use_gate, threshold_kind)
'''
import importlib.util, sys
from pathlib import Path
Path(DEPLOY).mkdir(parents=True, exist_ok=True); LIB_PATH = Path(DEPLOY)/'sexpred_lib.py'; LIB_PATH.write_text(LIB_SRC)
spec = importlib.util.spec_from_file_location('sexpred_lib', LIB_PATH); L = importlib.util.module_from_spec(spec); sys.modules['sexpred_lib'] = L; spec.loader.exec_module(L)
print('sexpred_lib', L.LIB_VERSION, '->', LIB_PATH)

## 1. Load inputs from Notebook 1

In [ ]:
import json, pickle, shutil, time, warnings, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x
warnings.filterwarnings('ignore')
BLUE, GREY, INK = '#2F6FB0', '#8A8F98', '#333333'
INPUTS, CVC, FINALC = Path(DEPLOY)/'inputs', Path(DEPLOY)/'cv_cache', Path(DEPLOY)/'final_components'
PKG = Path(DEPLOY)/'packages'/PACKAGE
for p in (CVC, FINALC, PKG): p.mkdir(parents=True, exist_ok=True)
db_path = next(p for p in (Path(ROOT)/'metadata'/'db.csv', Path(ROOT)/'db.csv') if p.exists())
db = pd.read_csv(db_path).drop_duplicates(['patient_id', 'step'])[['patient_id', 'step', 'sex']]
F = {lab: pd.read_csv(INPUTS / f'features_{lab}.csv', float_precision='round_trip') for lab in LENGTHS}
S = F['full'][['key', 'patient_id', 'step']].merge(db, on=['patient_id', 'step'])
conflict = S.groupby('patient_id').sex.nunique().loc[lambda s: s > 1].index.tolist()
S = S[~S.patient_id.isin(conflict)].sort_values(['patient_id', 'step']).reset_index(drop=True)
KEYS = S.key.values; PIDS = S.patient_id.values; Y_S = (S.sex == 'M').astype(int).values; FLAG = (S.step == 'after').values.astype(np.float32)
PEOPLE = np.array(sorted(set(PIDS))); YP = S.groupby('patient_id').sex.first().reindex(PEOPLE).eq('M').astype(int).values
POS_OF = pd.Series(np.arange(len(PEOPLE)), index=PEOPLE); SP = POS_OF.loc[PIDS].values      # session -> person position
for lab in LENGTHS: F[lab] = F[lab].set_index('key').reindex(KEYS)
M12, P4, OK = {}, {}, {}
for lab in LENGTHS:
    with np.load(INPUTS / f'signals_{lab}.npz') as z:
        raws = [z[k] if k in z.files else None for k in KEYS]
    vv = [L.views(r) if r is not None else (None, None) for r in tqdm(raws, desc=f'signals {lab}')]; del raws
    M12[lab] = [v[0] for v in vv]; P4[lab] = [v[1] for v in vv]
    OK[lab] = np.array([L.usable(F[lab].iloc[i].to_dict(), M12[lab][i]) for i in range(len(KEYS))])
print(f'Excluded for conflicting sex labels: {conflict}')
print(f'{len(KEYS)} sessions, {len(PEOPLE)} people ({YP.sum()} men); always-female accuracy {1 - YP.mean():.3f}')
print('Usable sessions by length: ' + ', '.join(f'{lab} {OK[lab].mean():.1%}' for lab in LENGTHS))
SIG_KEY = dict(n_people=len(PEOPLE), repeats=CV_REPEATS, seeds=CNN_SEEDS, epochs=EPOCHS, kernels=N_KERNELS, lengths=LENGTHS, quick=QUICK, lib=L.LIB_VERSION)
TAG = f"r{CV_REPEATS}_s{CNN_SEEDS}_e{EPOCHS}_k{N_KERNELS}_n{len(PEOPLE)}" + ('_quick' if QUICK else '')

## 2. Cross-validation of each model part
Folds are drawn per person (5 folds, stratified by sex), the same as Weeks 9–10. Each part is trained on the training people's **full-length** sessions and scores the test people's sessions at every length. Results are saved in `sex_deploy/cv_cache` and shared between the two package notebooks, so the second one only trains what is new.

| Part | Trained on | Scores |
|---|---|---|
| `cnn` | 12-channel rPPG, all training sessions | each test session |
| `rc` | random-conv features of the 4 POS channels | each test session |
| `tab` | rPPG + pulse-shape features | before/after/change per person (two-session model) and each session (single-session model) |
| `app` | skin-colour features | as `tab` |
| `gate` | quality features, label = rPPG HR off by > 5 BPM from contact HR over the same seconds | each test session |

In [ ]:
def person_folds(r):
    f = np.zeros(len(PEOPLE), int)
    for k, (_, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=r).split(np.zeros(len(PEOPLE)), YP, PEOPLE)): f[te] = k
    return f
FOLDS = {r: person_folds(r) for r in range(CV_REPEATS)}
def cached(name, fn):
    p = CVC / f'{name}_{TAG}.pkl'
    if p.exists(): return pickle.load(open(p, 'rb'))
    out = fn(); pickle.dump(out, open(p, 'wb')); return out
def sess_split(r, k):
    te_p = np.where(FOLDS[r] == k)[0]; te = np.where(np.isin(SP, te_p))[0]; tr = np.where(~np.isin(SP, te_p))[0]; return tr, te
def two_rows(lab, cols):
    rows = []
    for pid in PEOPLE:
        fb, fa = (F[lab].loc[f'{pid}_{s}'].to_dict() if f'{pid}_{s}' in F[lab].index else None for s in ('before', 'after'))
        rows.append(L.two_session_row(fb, fa, cols))
    return np.array(rows)
def single_rows(lab, cols): return np.array([L.single_session_row(F[lab].iloc[i].to_dict(), FLAG[i], cols) for i in range(len(KEYS))])

def cv_session_model(make, X):
    # X: {length: session matrix}. Trains on full-length usable training sessions. Returns {(r, length): session probabilities}.
    out = {}
    for r in range(CV_REPEATS):
        for k in range(5):
            tr, te = sess_split(r, k); tr = tr[OK['full'][tr]]; m = make().fit(X['full'][tr], Y_S[tr])
            for lab in LENGTHS:
                out.setdefault((r, lab), np.full(len(KEYS), np.nan))[te] = m.predict_proba(X[lab][te])[:, 1]
    return out
def cv_person_model(make, X):
    out = {}
    for r in range(CV_REPEATS):
        for k in range(5):
            tr, te = np.where(FOLDS[r] != k)[0], np.where(FOLDS[r] == k)[0]; m = make().fit(X['full'][tr], YP[tr])
            for lab in LENGTHS:
                out.setdefault((r, lab), np.full(len(PEOPLE), np.nan))[te] = m.predict_proba(X[lab][te])[:, 1]
    return out
CV = {}
t0 = time.time()
def cv_gate():
    X = {lab: F[lab][L.GATE_COLS].values.astype(float) for lab in LENGTHS}
    bad = {lab: ((F[lab].rppg_hr - F[lab].ppg_hr).abs() > 5).values for lab in LENGTHS}; lab_ok = {lab: (F[lab].ppg_hr.notna() & F[lab].rppg_hr.notna()).values for lab in LENGTHS}
    out = {}
    for r in range(CV_REPEATS):
        for k in range(5):
            tr, te = sess_split(r, k)
            Xtr = np.vstack([X[lab][tr][lab_ok[lab][tr]] for lab in LENGTHS]); ytr = np.concatenate([bad[lab][tr][lab_ok[lab][tr]] for lab in LENGTHS])
            g = L.make_gate().fit(Xtr, ytr)
            for lab in LENGTHS: out.setdefault((r, lab), np.full(len(KEYS), np.nan))[te] = g.predict_proba(X[lab][te])[:, 1]
    return out
CV['gate'] = cached('gate', cv_gate); print(f'gate done ({time.time() - t0:.0f} s)')
RC_X = {lab: cached(f'rcfeat_{lab}', lambda lab=lab: L.rc_session_features([P4[lab][i] if OK[lab][i] else None for i in range(len(KEYS))], N_KERNELS)) for lab in LENGTHS}
CV['rc'] = cached('rc', lambda: cv_session_model(L.make_rc_model, RC_X)); print(f'random conv done ({time.time() - t0:.0f} s)')
for comp, cols in (('tab', L.TAB_COLS), ('app', L.APP_COLS)):
    if comp not in COMPONENTS: continue
    CV[f'{comp}_two'] = cached(f'{comp}_two', lambda cols=cols: cv_person_model(L.make_hgb, {lab: two_rows(lab, cols) for lab in LENGTHS}))
    CV[f'{comp}_single'] = cached(f'{comp}_single', lambda cols=cols: cv_session_model(L.make_hgb, {lab: single_rows(lab, cols) for lab in LENGTHS}))
    print(f'{comp} done ({time.time() - t0:.0f} s)')

### CNN in cross-validation
`CV_REPEATS` × 5 folds × `CNN_SEEDS` fits. Each finished fold is saved, so a disconnect resumes at the next fold.

In [ ]:
import torch
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', DEV)
if str(DEV) == 'cpu': print('WARNING: no GPU. CNN training will take many hours.')
def cv_cnn():
    out = {}
    for r in range(CV_REPEATS):
        for k in range(5):
            fp = CVC / f'cnn_fold_r{r}_k{k}_{TAG}.pkl'
            if fp.exists(): part = pickle.load(open(fp, 'rb'))
            else:
                tr, te = sess_split(r, k); tr = tr[OK['full'][tr]]; part = {}
                for s in range(CNN_SEEDS):
                    net = L.train_cnn(M12['full'], FLAG, Y_S, PIDS, tr, 1000 * r + 10 * k + s, DEV, epochs=EPOCHS, patience=PATIENCE)
                    for lab in LENGTHS:
                        part.setdefault(lab, []).append(np.array([L.cnn_session_logit(net, M12[lab][i], FLAG[i], DEV) if OK[lab][i] else np.nan for i in te]))
                part = dict(te=te, logits=part); pickle.dump(part, open(fp, 'wb'))
            for lab in LENGTHS:
                out.setdefault((r, lab), np.full(len(KEYS), np.nan))[part['te']] = L.sigmoid(np.nanmean(part['logits'][lab], 0))
            print(f'CNN repeat {r} fold {k} done')
    return out
CV['cnn'] = cached('cnn', cv_cnn)

## 3. Deployment tests
For each person, the sessions available in the scenario (both, before only or after only), at the given length, minus any rejected by the gate. Then:
- CNN and random conv: average of the session probabilities;
- tabular parts: the two-session model if both sessions remain, otherwise the single-session model;
- fusion: mean log-odds of the parts.

Accuracy uses a calibration and threshold chosen on the other four-fifths of people (cross-fitted), so no test person sets their own threshold. Confidence intervals resample people.

In [ ]:
def fused(r, lab, scenario, gate):
    want = {'two sessions': ('before', 'after'), 'before only': ('before',), 'after only': ('after',)}[scenario]
    use = OK[lab] & np.isin(S.step.values, want)
    if gate: use &= np.nan_to_num(CV['gate'][(r, lab)], nan=1) < GATE_THRESHOLD
    rows = []
    for j, pid in enumerate(PEOPLE):
        idx = np.where((SP == j) & use)[0]
        if not len(idx): continue
        steps = set(S.step.values[idx]); comp = {}
        for c in ('cnn', 'rc'):
            if c in COMPONENTS: comp[c] = np.mean(CV[c][(r, lab)][idx])
        for c in ('tab', 'app'):
            if c in COMPONENTS: comp[c] = CV[f'{c}_two'][(r, lab)][j] if steps == {'before', 'after'} else np.mean(CV[f'{c}_single'][(r, lab)][idx])
        rows.append(dict(pid=pid, j=j, y=YP[j], z=np.mean(L.logit(list(comp.values()))), **{f'p_{c}': v for c, v in comp.items()}))
    return pd.DataFrame(rows)
def best_thr(p, y, kind):
    c = np.unique(np.quantile(p, np.linspace(.01, .99, 99)))
    sc = [((p >= t) == y).mean() if kind == 'accuracy' else ((p[y == 1] >= t).mean() + (p[y == 0] < t).mean()) / 2 for t in c]
    return c[int(np.argmax(sc))]
def crossfit_decisions(z, y, seed=0):
    out = {k: np.zeros(len(y), bool) for k in ('balanced', 'accuracy')}; cal = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(z.reshape(-1, 1), y):
        m = LogisticRegression(C=1e4).fit(z[tr, None], y[tr]); ptr, pte = m.predict_proba(z[tr, None])[:, 1], m.predict_proba(z[te, None])[:, 1]; cal[te] = pte
        for k in out: out[k][te] = pte >= best_thr(ptr, y[tr], k)
    return out, cal
def evaluate(scenario, lab, gate, comps=None):
    frames = {r: fused(r, lab, scenario, gate) for r in range(CV_REPEATS)}
    if comps:  # a single part only
        for r, d in frames.items(): d['z'] = L.logit(d[f'p_{comps}'])
    per = []; dec = {}
    for r, d in frames.items():
        y, z = d.y.values, d.z.values; o, cal = crossfit_decisions(z, y, r); dec[r] = (d.j.values, y, z, o)
        ece = sum(abs(cal[b].mean() - y[b].mean()) * b.mean() for b in [(cal >= lo) & (cal < lo + .1 + 1e-9 * (lo > .85)) for lo in np.arange(0, 1, .1)] if b.any())
        per.append(dict(coverage=len(d) / len(PEOPLE), auc=roc_auc_score(y, z), acc_bal=(o['balanced'] == y).mean(), recall_m_bal=o['balanced'][y == 1].mean(),
                        recall_f_bal=(~o['balanced'][y == 0]).mean(), acc_acc=(o['accuracy'] == y).mean(), recall_m_acc=o['accuracy'][y == 1].mean(),
                        recall_f_acc=(~o['accuracy'][y == 0]).mean(), always_female=1 - y.mean(), ece=ece))
    res = pd.DataFrame(per).mean().to_dict(); rg = np.random.default_rng(0); bs = []
    for _ in range(N_BOOT):
        pick = rg.integers(0, len(PEOPLE), len(PEOPLE)); w = np.bincount(pick, minlength=len(PEOPLE)); vals = []
        for r, (jj, y, z, o) in dec.items():
            wt = w[jj]; ix = np.repeat(np.arange(len(jj)), wt)
            if len(ix) and 0 < y[ix].sum() < len(ix): vals.append((roc_auc_score(y[ix], z[ix]), (o['balanced'][ix] == y[ix]).mean(), (o['accuracy'][ix] == y[ix]).mean()))
        if vals: bs.append(np.mean(vals, 0))
    bs = np.array(bs)
    for i, k in enumerate(('auc', 'acc_bal', 'acc_acc')): res[k + '_lo'], res[k + '_hi'] = np.percentile(bs[:, i], [2.5, 97.5])
    return res
RESULTS = []
for scen in ('two sessions', 'before only', 'after only'):
    for lab in LENGTHS:
        for gate in (False, True):
            RESULTS.append(dict(scenario=scen, length=lab, gate=gate, **evaluate(scen, lab, gate)))
R = pd.DataFrame(RESULTS); R.round(4).to_csv(PKG / 'deployment_tests.csv', index=False)
cols = ['scenario', 'length', 'gate', 'coverage', 'auc', 'auc_lo', 'auc_hi', 'acc_bal', 'acc_bal_lo', 'acc_bal_hi', 'recall_m_bal', 'recall_f_bal', 'acc_acc', 'recall_m_acc', 'recall_f_acc', 'always_female', 'ece']
pd.set_option('display.width', 250, 'display.max_columns', 40); display(R[cols].round(3))
PARTS = pd.DataFrame([dict(part=c, **evaluate('two sessions', 'full', False, comps=c)) for c in COMPONENTS])
print('Each part alone (two sessions, full length, no gate):'); display(PARTS[['part', 'auc', 'auc_lo', 'auc_hi', 'acc_bal', 'recall_m_bal', 'recall_f_bal']].round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
order = [l for l in ['30s', '60s', '120s', 'full'] if l in LENGTHS]; x = np.arange(len(order))
for scen, ls in (('two sessions', '-'), ('before only', '--'), ('after only', ':')):
    d = R[(R.scenario == scen) & R.gate].set_index('length').reindex(order)
    ax[0].errorbar(x, d.auc, yerr=[d.auc - d.auc_lo, d.auc_hi - d.auc], ls=ls, marker='o', color=BLUE if scen == 'two sessions' else INK, lw=2, label=scen, capsize=3)
    ax[1].plot(x, d.acc_bal, ls=ls, marker='o', color=BLUE if scen == 'two sessions' else INK, lw=2, label=scen)
ax[1].axhline(1 - YP.mean(), color=GREY, ls='--', lw=1); ax[1].text(x[0], 1 - YP.mean() + .005, 'always female', color=GREY, fontsize=8)
for a, t in zip(ax, ('AUC (95% CI), with quality gate', 'Accuracy, balanced threshold, with quality gate')):
    a.set_xticks(x, order); a.set(xlabel='Recording length used', title=t); a.spines[['top', 'right']].set_visible(False); a.legend(frameon=False)
plt.tight_layout(); plt.savefig(PKG / 'deployment_tests.png', dpi=150); plt.show()

## 4. Train the final models on everyone
Shared parts (`cnn`, `rc`, `tab`, `gate`) are saved in `sex_deploy/final_components` and reused by the other package notebook. The calibration and thresholds come from the cross-validation scores at full length with the gate on:
- **two-session** mode uses the two-session scenario;
- **single-session** mode pools the before-only and after-only scenarios.

In [ ]:
def final(name, fn):
    p = FINALC / f'{name}_{TAG}.joblib'
    if p.exists(): return joblib.load(p)
    m = fn(); joblib.dump(m, p); return m
okf = OK['full']
FM = {}
FM['gate'] = final('gate', lambda: L.make_gate().fit(np.vstack([F[l][L.GATE_COLS].values[(F[l].ppg_hr.notna() & F[l].rppg_hr.notna()).values] for l in LENGTHS]),
                                                    np.concatenate([((F[l].rppg_hr - F[l].ppg_hr).abs() > 5).values[(F[l].ppg_hr.notna() & F[l].rppg_hr.notna()).values] for l in LENGTHS])))
FM['rc'] = final('rc', lambda: L.make_rc_model().fit(RC_X['full'][okf], Y_S[okf]))
for comp, cols in (('tab', L.TAB_COLS), ('app', L.APP_COLS)):
    if comp in COMPONENTS:
        FM[f'{comp}_two'] = final(f'{comp}_two', lambda cols=cols: L.make_hgb().fit(two_rows('full', cols), YP))
        FM[f'{comp}_single'] = final(f'{comp}_single', lambda cols=cols: L.make_hgb().fit(single_rows('full', cols)[okf], Y_S[okf]))
CNN_FILES = []
for s in range(FINAL_CNN_SEEDS):
    p = FINALC / f'cnn_seed{s}_{TAG}.pt'
    if not p.exists():
        net = L.train_cnn(M12['full'], FLAG, Y_S, PIDS, np.where(okf)[0], 7000 + s, DEV, epochs=EPOCHS, patience=PATIENCE); torch.save(net.state_dict(), p)
    CNN_FILES.append(p); print(f'final CNN {s + 1}/{FINAL_CNN_SEEDS} ready')
CAL, THR = {}, {}
for mode, scens in (('two', ['two sessions']), ('single', ['before only', 'after only'])):
    z = np.concatenate([fused(r, 'full', sc, True).z.values for r in range(CV_REPEATS) for sc in scens]); y = np.concatenate([fused(r, 'full', sc, True).y.values for r in range(CV_REPEATS) for sc in scens])
    CAL[mode] = LogisticRegression(C=1e4).fit(z[:, None], y); p = CAL[mode].predict_proba(z[:, None])[:, 1]
    THR[mode] = {k: float(best_thr(p, y, k)) for k in ('balanced', 'accuracy')}
print('Thresholds on calibrated P(male):', THR)

## 5. Write the package

In [ ]:
for f in PKG.glob('*'):
    if f.suffix in ('.joblib', '.pt', '.py', '.json', '.md'): f.unlink()
shutil.copy(LIB_PATH, PKG / 'sexpred_lib.py')
for name in ['gate', 'rc'] + [f'{c}_{m}' for c in ('tab', 'app') if c in COMPONENTS for m in ('two', 'single')]: joblib.dump(FM[name], PKG / f'{name}.joblib')
joblib.dump(CAL, PKG / 'calibrators.joblib')
for s, p in enumerate(CNN_FILES): shutil.copy(p, PKG / f'cnn_seed{s}.pt')
main = R[(R.scenario == 'two sessions') & (R.length == 'full') & R.gate].iloc[0]; single = R[(R.scenario == 'before only') & (R.length == 'full') & R.gate].iloc[0]
CFG = dict(package=PACKAGE, components=COMPONENTS, lib_version=L.LIB_VERSION, gate_threshold=GATE_THRESHOLD, thresholds=THR, default_threshold=DEFAULT_THRESHOLD,
           n_kernels=N_KERNELS, cnn_copies=len(CNN_FILES), trained_on=dict(people=int(len(PEOPLE)), men=int(YP.sum()), dataset='MCD-rPPG front camera (FullHDwebcam)'),
           cv=dict(two_sessions_full_gate=dict(auc=main.auc, auc_ci=[main.auc_lo, main.auc_hi], acc_balanced=main.acc_bal, coverage=main.coverage),
                   before_only_full_gate=dict(auc=single.auc, auc_ci=[single.auc_lo, single.auc_hi], acc_balanced=single.acc_bal, coverage=single.coverage)),
           created=pd.Timestamp.now().isoformat(timespec='seconds'), quick_test=QUICK)
(PKG / 'config.json').write_text(json.dumps(CFG, indent=1, default=float))
(PKG / 'predict.py').write_text('''"""Command line: python predict.py --before before.avi [--before-meta before.txt] [--after after.avi] [--after-meta after.txt]"""
import argparse, json, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent))
from sexpred_lib import SexPredictor
ap = argparse.ArgumentParser()
for a in ('--before', '--after', '--before-meta', '--after-meta'): ap.add_argument(a)
ap.add_argument('--threshold', choices=['balanced', 'accuracy']); ap.add_argument('--no-gate', action='store_true')
a = ap.parse_args()
out = SexPredictor(Path(__file__).parent).predict(a.before, a.after, a.before_meta, a.after_meta, use_gate=not a.no_gate, threshold_kind=a.threshold)
print(json.dumps(out, indent=1, default=str))
''')
def md(d):
    d = d.round(3).astype(str); return '\n'.join(['| ' + ' | '.join(d.columns) + ' |', '|' + '---|' * d.shape[1]] + ['| ' + ' | '.join(r) + ' |' for r in d.values])
tbl = md(R[R.gate & (R.length == 'full')][['scenario', 'coverage', 'auc', 'acc_bal', 'recall_m_bal', 'recall_f_bal', 'acc_acc']])
tbl_len = md(R[R.gate & (R.scenario == 'two sessions')][['length', 'coverage', 'auc', 'acc_bal', 'recall_m_bal', 'recall_f_bal']])
(PKG / 'MODEL_CARD.md').write_text(f"""# {PACKAGE}: sex prediction from face video

Four parts averaged on the log-odds scale: Package A's three rPPG parts plus gradient boosting on skin-colour features measured from the same video. Skin colour is appearance, not physiology, and depends on camera colour settings and lighting.

## Use
```python
from sexpred_lib import SexPredictor
p = SexPredictor('{PKG}')
p.predict(before_video='before.avi', before_meta='before.txt')                       # one recording
p.predict(before_video='b.avi', after_video='a.avi', before_meta='b.txt', after_meta='a.txt')   # before and after 15 squats
```
Returns `probability_male` (calibrated to a cohort that is one-third men), `prediction` ('M', 'F', or None when no session passes the quality gate), the part scores and per-session quality. The meta file holds per-frame timestamps; without it the video's own frame rate is used.

## Cross-validated performance (MCD-rPPG, {len(PEOPLE)} people, {CV_REPEATS} x 5-fold by person, quality gate on)
Full-length recordings:

{tbl}

Two sessions, by recording length:

{tbl_len}

`acc_bal`: accuracy at the balanced threshold; `acc_acc`: at the accuracy threshold. Always answering "female" scores {1 - YP.mean():.3f}.

## Limits
- Trained and tested on one dataset: one site, one camera (FullHD webcam, about 30 fps), mostly ages 18-29, 3-minute seated recordings. Not validated on other cameras, lighting or populations.
- Part of the signal reflects signal-quality and appearance differences between men and women, not only the heartbeat (Week 10).
- The quality gate was trained against a contact sensor on MCD; at use time it needs no sensor.
""")
print('Package written to', PKG); print(sorted(f.name for f in PKG.iterdir()))

## 6. Check the package from disk
1. **Traces path.** The package is loaded fresh from Drive and scores 20 people from their stored traces. The same people are scored inside this notebook from the prepared inputs and the final models. The two must agree; a difference means the deployed code path differs from the training path.
2. **Video path.** A few people are predicted straight from `raw/video` (re-extracting the face traces). Small differences from step 1 come only from re-running the face detector.

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location('pkg_lib', PKG / 'sexpred_lib.py'); PL = importlib.util.module_from_spec(spec); spec.loader.exec_module(PL)

In [ ]:
PRED = PL.SexPredictor(PKG, device=str(DEV))
fr = pd.read_csv(db_path).query("camera == 'FullHDwebcam'").drop_duplicates(['patient_id', 'step']).set_index(['patient_id', 'step'])
def notebook_score(j):
    idx = [i for i in np.where(SP == j)[0] if OK['full'][i]]
    xs = [dict(step=S.step.values[i], feats=F['full'].iloc[i].to_dict(), m12=M12['full'][i], p4=P4['full'][i]) for i in idx]
    return PRED.score_sessions(xs)
rows = []
for j in np.random.default_rng(1).choice(len(PEOPLE), min(20, len(PEOPLE)), replace=False):
    pid = PEOPLE[j]; a = notebook_score(j); ss = []
    for st in ('before', 'after'):
        r = fr.loc[(pid, st)]; ss.append(PRED.session_from_traces(PL.load_traces(Path(ROOT)/'roi_traces'/f"{Path(r['video']).name}.npz"), PL.parse_meta_timestamps(Path(ROOT)/'raw'/r['meta']), st))
    b = PRED.score_sessions(ss)
    rows.append(dict(person=pid, sex='M' if YP[j] else 'F', from_inputs=a['probability_male'], from_traces=b['probability_male'], pred=b['prediction'], mode=b['mode']))
CHK = pd.DataFrame(rows); CHK['abs_diff'] = (CHK.from_inputs - CHK.from_traces).abs(); display(CHK.round(4))
print(f"Largest difference, inputs vs traces: {CHK.abs_diff.max():.2e}", '-> OK' if CHK.abs_diff.max() < 1e-3 else '-> CHECK: the deployed path differs from training')
VID = []
if N_VIDEO_CHECK:
    for pid in CHK.person[:N_VIDEO_CHECK]:
        vids = {st: Path(ROOT)/'raw'/fr.loc[(pid, st)]['video'] for st in ('before', 'after')}
        if not all(v.exists() for v in vids.values()): continue
        t0 = time.time(); tmp = Path('/content/vid_tmp' if Path('/content').exists() else './vid_tmp'); tmp.mkdir(exist_ok=True); loc = {}
        for st, v in vids.items(): loc[st] = tmp / v.name; shutil.copy(v, loc[st])
        o = PRED.predict(loc['before'], loc['after'], Path(ROOT)/'raw'/fr.loc[(pid, 'before')]['meta'], Path(ROOT)/'raw'/fr.loc[(pid, 'after')]['meta'])
        for p in loc.values(): p.unlink()
        VID.append(dict(person=pid, from_traces=float(CHK.set_index('person').loc[pid, 'from_traces']), from_video=o['probability_male'], pred=o['prediction'], seconds=round(time.time() - t0)))
    VID = pd.DataFrame(VID); display(VID.round(4))
CHK.to_csv(PKG / 'package_check.csv', index=False)
if len(VID): VID.to_csv(PKG / 'package_check_video.csv', index=False)

## 7. Summary

In [ ]:
print(f"Package: {PKG}")
for scen in ('two sessions', 'before only', 'after only'):
    d = R[(R.scenario == scen) & (R.length == 'full') & R.gate].iloc[0]
    print(f"{scen:13s} full length, gate on: AUC {d.auc:.3f} [{d.auc_lo:.3f}, {d.auc_hi:.3f}], accuracy (balanced) {d.acc_bal:.3f} [{d.acc_bal_lo:.3f}, {d.acc_bal_hi:.3f}], "
          f"recall M {d.recall_m_bal:.2f} F {d.recall_f_bal:.2f}; coverage {d.coverage:.1%}")
d = R[(R.scenario == 'two sessions') & R.gate]
print('Two sessions by length (AUC): ' + ', '.join(f"{r.length} {r.auc:.3f}" for r in d.itertuples()))
print(f"Always-female accuracy {1 - YP.mean():.3f}. Package check: traces path max difference {CHK.abs_diff.max():.1e}" + (f"; video path checked on {len(VID)} people" if len(VID) else ''))